# Detection-criteria adherence across the sweep

This notebook audits whether serialized detected lightcurves satisfy criteria that can be checked from the stored timestamps and counts. It does not rerun simulations.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('results/detection_criteria_sweep')
QUALITY = ROOT / 'lightcurve_quality.csv'
if not QUALITY.exists():
    raise FileNotFoundError(f'Run lightcurve_quality_analysis.py first: {QUALITY}')
quality = pd.read_csv(QUALITY)
quality.head()

,file,source_index,z,t_exp,n_positive,n_non_detections,n_bands,bands,duplicate_positive_times,positive_span_days,...,positive_span_exceeds_14_days,all_observation_span_exceeds_14_days,criteria_snr_threshold,criteria_snr_threshold_secondary,criteria_min_detections,criteria_min_detections_primary,criteria_min_time_separation_hours,criteria_min_fade_rate,criteria_min_galactic_lat,band_counts
0,results/detection_criteria_sweep/snr4_snr22_nd...,0,0.02857,60154.698,9,100,2,g;r,0,88.89029,...,True,True,4.0,2.0,2,1,12.0,0.3,0.0,"{""g"": 2, ""r"": 7}"
1,results/detection_criteria_sweep/snr4_snr22_nd...,1,0.06014,58525.968,1,435,1,g,0,0.00000,...,False,True,4.0,2.0,2,1,12.0,0.3,0.0,"{""g"": 1}"
2,results/detection_criteria_sweep/snr4_snr22_nd...,2,0.04193,59464.779,11,220,2,g;r,0,49.94175,...,True,True,4.0,2.0,2,1,12.0,0.3,0.0,"{""g"": 7, ""r"": 4}"
3,results/detection_criteria_sweep/snr4_snr22_nd...,3,0.02556,60691.361,3,74,2,g;r,0,22.00574,...,True,True,4.0,2.0,2,1,12.0,0.3,0.0,"{""g"": 1, ""r"": 2}"
4,results/detection_criteria_sweep/snr4_snr22_nd...,4,0.03360,60263.142,12,109,2,g;r,0,62.72165,...,True,True,4.0,2.0,2,1,12.0,0.3,0.0,"{""g"": 6, ""r"": 6}"


## Measurable adherence checks

The audit uses the detected positive timestamps stored in each result. A source passes the checks below when its stored lightcurve has enough positive points, fits within the configured timespan, and has no positive-detection gap shorter than the configured minimum separation.

In [2]:
numeric_columns = [
    'n_positive', 'positive_span_days', 'configured_max_timespan_days',
    'min_positive_gap_days', 'criteria_min_detections',
    'criteria_min_time_separation_hours', 'criteria_min_detections_primary',
]
quality[numeric_columns] = quality[numeric_columns].apply(pd.to_numeric, errors='coerce')
quality['min_positive_gap_hours'] = quality['min_positive_gap_days'] * 24.0
quality['passes_min_detections'] = quality['n_positive'] >= quality['criteria_min_detections']
quality['passes_max_timespan'] = quality['positive_span_days'] <= quality['configured_max_timespan_days']
quality['passes_min_time_separation'] = (
    (quality['n_positive'] < 2)
    | (quality['min_positive_gap_hours'] >= quality['criteria_min_time_separation_hours'])
)
quality['passes_measurable_criteria'] = quality[[
    'passes_min_detections', 'passes_max_timespan', 'passes_min_time_separation'
]].all(axis=1)
quality[['passes_min_detections', 'passes_max_timespan', 'passes_min_time_separation', 'passes_measurable_criteria']].mean()

passes_min_detections         0.733204
passes_max_timespan           0.307814
passes_min_time_separation    0.516021
passes_measurable_criteria    0.036843
dtype: float64

In [3]:
grid_columns = [
    'criteria_snr_threshold', 'criteria_snr_threshold_secondary',
    'criteria_min_detections', 'criteria_min_detections_primary',
    'configured_max_timespan_days', 'criteria_min_time_separation_hours',
    'criteria_min_fade_rate', 'criteria_min_galactic_lat',
]
adherence_summary = (
    quality.groupby(grid_columns, dropna=False)
    .agg(
        detected_sources=('source_index', 'size'),
        min_detections_pass_rate=('passes_min_detections', 'mean'),
        max_timespan_pass_rate=('passes_max_timespan', 'mean'),
        min_separation_pass_rate=('passes_min_time_separation', 'mean'),
        measurable_criteria_pass_rate=('passes_measurable_criteria', 'mean'),
        median_span_days=('positive_span_days', 'median'),
        median_gap_hours=('min_positive_gap_hours', 'median'),
    )
    .reset_index()
)
adherence_summary.sort_values('measurable_criteria_pass_rate').head(20)

,criteria_snr_threshold,criteria_snr_threshold_secondary,criteria_min_detections,criteria_min_detections_primary,configured_max_timespan_days,criteria_min_time_separation_hours,criteria_min_fade_rate,criteria_min_galactic_lat,detected_sources,min_detections_pass_rate,max_timespan_pass_rate,min_separation_pass_rate,measurable_criteria_pass_rate,median_span_days,median_gap_hours
2107,6.0,3.0,3,2,7.0,12.0,0.3,0.0,11,0.636364,0.090909,0.454545,0.0,66.966470,1.61376
2120,6.0,3.0,3,2,14.0,12.0,0.3,15.0,15,0.933333,0.066667,0.266667,0.0,73.866390,0.11736
2121,6.0,3.0,3,2,14.0,12.0,0.6,0.0,19,0.789474,0.263158,0.578947,0.0,53.895830,0.52416
1243,5.0,3.0,3,1,14.0,12.0,0.3,0.0,20,0.650000,0.250000,0.550000,0.0,51.407990,1.45272
1245,5.0,3.0,3,1,14.0,12.0,0.6,0.0,19,0.736842,0.315789,0.526316,0.0,49.757660,0.84960
1246,5.0,3.0,3,1,14.0,12.0,0.6,15.0,20,0.850000,0.300000,0.500000,0.0,47.393790,1.47744
1250,5.0,3.0,3,1,21.0,3.0,0.3,15.0,34,0.794118,0.176471,0.470588,0.0,53.940135,1.51212
1256,5.0,3.0,3,1,21.0,12.0,0.3,15.0,43,0.790698,0.186047,0.511628,0.0,72.836370,1.35408
1232,5.0,3.0,3,1,7.0,12.0,0.3,15.0,8,0.875000,0.250000,0.500000,0.0,56.457175,1.17504
1233,5.0,3.0,3,1,7.0,12.0,0.6,0.0,8,0.875000,0.125000,0.500000,0.0,70.917335,49.03848


## Violations and diagnostics

Rows below show the worst measurable adherence combinations. The `all_observation_span` check is diagnostic only: non-detections may be stored outside the qualifying positive-detection window.

In [4]:
violations = quality.loc[~quality['passes_measurable_criteria']].copy()
violation_counts = pd.Series({
    'minimum detections': (~quality['passes_min_detections']).sum(),
    'maximum timespan': (~quality['passes_max_timespan']).sum(),
    'minimum time separation': (~quality['passes_min_time_separation']).sum(),
    'any measurable criterion': (~quality['passes_measurable_criteria']).sum(),
})
print(violation_counts)
violations.sort_values(['positive_span_days', 'min_positive_gap_hours'], ascending=False)[[
    'file', 'source_index', 'n_positive', 'positive_span_days',
    'configured_max_timespan_days', 'min_positive_gap_hours',
    'criteria_min_time_separation_hours', 'criteria_min_detections'
]].head(25)

minimum detections           47468
maximum timespan            123153
minimum time separation      86109
any measurable criterion    171364
dtype: int64


,file,source_index,n_positive,positive_span_days,configured_max_timespan_days,min_positive_gap_hours,criteria_min_time_separation_hours,criteria_min_detections
44942,results/detection_criteria_sweep/snr4_snr24_nd...,4,35,104.84413,7.0,0.02208,12.0,2
60842,results/detection_criteria_sweep/snr4_snr24_nd...,80,20,104.84271,7.0,0.02256,3.0,3
110990,results/detection_criteria_sweep/snr5_snr24_nd...,62,3,104.77905,7.0,1196.24448,3.0,2
159336,results/detection_criteria_sweep/snr6_snr24_nd...,45,8,104.77588,21.0,0.56688,3.0,2
146608,results/detection_criteria_sweep/snr6_snr23_nd...,18,2,104.72227,14.0,2513.33448,3.0,3
160426,results/detection_criteria_sweep/snr6_snr24_nd...,1,7,104.68557,3.0,0.01104,12.0,2
45930,results/detection_criteria_sweep/snr4_snr24_nd...,216,28,104.01222,14.0,0.14208,12.0,2
63048,results/detection_criteria_sweep/snr4_snr24_nd...,28,12,103.91877,21.0,1.35768,12.0,4
45016,results/detection_criteria_sweep/snr4_snr24_nd...,56,18,103.90893,7.0,0.62160,12.0,2
56875,results/detection_criteria_sweep/snr4_snr24_nd...,14,14,103.90308,14.0,0.01104,3.0,3


## Criteria that require richer serialization

The CSV can report configured SNR thresholds, fade rates, Galactic latitude, and primary-detection requirements, but it cannot prove those tests were applied because the serialized source records do not include per-observation SNR, peak/rise/fade classification, Galactic latitude, or primary/secondary labels. Treat those fields as configuration metadata until the simulator serializes the detector decision details.